# Stage 11: Supervised Rat-Space Exercise Subspace Probe

> **References shown: all tissues (16)**: the refs4 references (`references_v4/`, jointly re-annotated) plus SPLEEN (GSE306764) and OVARY (GSE343214); default since 2026-11-01, commit c3b27c7. **Model: retrain v2** (`checkpoint-591763`). Outputs are read from the `_refs4all` folders. The 14-tissue refs4 version of this notebook is in git history at commit `542b454`; refs4-vs-all-tissues is `deconvolution/ALL_TISSUE_RERUN_RESULTS.md`, July-vs-refs4 is `deconvolution/REFS4_RERUN_RESULTS.md`, and April-vs-v2 is `docs/RETRAIN_V2_RESULTS.md`. The probe tables are the all-tissue gate by the gate method of 10 stratified 5-fold splits and 10,000 permutations (`gate_rcv10/`).

## Stage Objective

Test whether exercise exposure is encoded in the rat GeneCompass latent space at the tissue-by-cell-type level.

## Workflow

| Step | Operation | Analytical purpose |
|------|-----------|--------------------|
| 1 | Block definition | Construct analyzable tissue-by-cell-type blocks with aligned embeddings and metadata. |
| 2 | Supervised probe | Fit a one-component PLS model and quantify held-out treatment discrimination. |
| 3 | Permutation inference | Estimate block-level significance and control the false discovery rate across blocks. |
| 4 | Corroboration | Compare supervised-probe results with the independent Augur random-forest analysis. |

## Inputs and Outputs

| Component | Description |
|-----------|-------------|
| Inputs | Stage 9 rat-space embeddings, aligned metadata, and Stage 10 analysis context. |
| Primary outputs | Block-level probe statistics, corroboration tables, and the set of rat-space exercise hotspots. |

## Result Overview

The analysis evaluates 151 analysable blocks (of 197) and identifies 9 FDR-significant exercise hotspots for the Stage 12 transfer test.

The sections below present the computational evidence in execution order. All reported figures are generated from the recorded stage outputs rather than manually entered plotting values unless explicitly identified as configuration constants.


In [1]:
import os, pathlib
from nb_config import PROJECT as ROOT
from nb_config import GATE_DIR as GCI

files = {
    "subspace_probe.tsv":      GCI / "subspace_probe.tsv",
    "corroboration_merged.tsv":GCI / "corroboration_merged.tsv",
    "pheno_merge_test.tsv":    GCI.parent / "pheno_merge_test.tsv",
}
for name, p in files.items():
    stat = p.stat() if p.exists() else None
    size_kb = stat.st_size / 1024 if stat else 0
    lines = sum(1 for _ in open(p)) if p.exists() else 0
    status = "OK" if p.exists() else "MISSING"
    print(f"  [{status}] {name}: {lines} rows, {size_kb:.1f} KB")


config: (default config)
Stage 8: /depot/reese18/apps/motrpac-genecompass/data/deconvolution/results_refs4all_rebuild/motrpac
Stage 9: /depot/reese18/apps/motrpac-genecompass/data/deconvolution/genecompass_input_v2_refs4all_rebuild
16 tissues: BAT, BLOOD, CORTEX, HEART, HIPPOC, HYPOTH, KIDNEY, LIVER, LUNG, OVARY, SKMGN, SKMVL, SMLINT, SPLEEN, TESTES, WATSC
  [OK] subspace_probe.tsv: 198 rows, 42.8 KB
  [OK] corroboration_merged.tsv: 198 rows, 33.7 KB
  [OK] pheno_merge_test.tsv: 198 rows, 12.1 KB


## Probe design and block structure

Each **block** = (tissue × cell-type) pair with ≥10 pseudo-cells.
- 16 tissues, 197 blocks; 151 analysable under the §6 rule (median per-sample library ≥ 1,000).
- Each block has one pseudo-cell per bulk sample: **n = 50** (2 sexes × 5 groups × 5 replicates; TESTES is male-only, n = 25; OVARY is female-only, n = 24).
- PLS-1 probe: stratified 5-fold CV, AUC computed on out-of-fold predictions and averaged over 10 independent splits.
- Permutation test: 10,000 label-shuffled runs per block, reusing the same splits; empirical p-value.
- Multiple testing: BH-FDR across the 151 analysable blocks → `q_sup_trained`.

**Four probe factors** tested per block:
1. `trained` — binary: trained (weeks 1/2/4/8) vs control
2. `dose(ord)` — ordinal: week 1/2/4/8 (training animals only)
3. `sex` — binary: male vs female (positive control, expected to be detectable)
4. `group` — four-class: trained 4w, trained 8w, control 4w, control 8w (omnibus)


In [2]:
import pandas as pd
from nb_config import GATE_DIR
GCI = str(GATE_DIR)
sp = pd.read_csv(f"{GCI}/subspace_probe.tsv", sep="\t")
q_values = pd.read_csv(
    f"{GCI}/corroboration_merged.tsv",
    sep="\t",
    usecols=["tissue", "cell_type", "q_sup_trained", "analysable"],
)
sp = sp.merge(q_values, on=["tissue", "cell_type"], how="left", validate="one_to_one")
sp = sp[sp["analysable"]].copy()   # §6 rule: blocks with median library < 1,000 are not analysed (no q value)
if sp["q_sup_trained"].isna().any():
    raise ValueError("Missing adjusted P values after merging Stage 11 result tables")

# q_sup_trained hotspot flag
sp["is_hotspot"] = sp["q_sup_trained"] < 0.05

summary = sp.groupby("tissue").agg(
    n_blocks         = ("tissue", "size"),
    n_hotspot        = ("is_hotspot", "sum"),
    median_sup_auc   = ("sup_trained_auc", "median"),
    max_sup_auc      = ("sup_trained_auc", "max"),
    median_sex_auc   = ("sup_sex_auc", "median"),
    median_dose_rho  = ("sup_dose_rho", "median"),
).reset_index()
summary["hotspot_pct"] = (summary["n_hotspot"] / summary["n_blocks"] * 100).round(1)
pd.set_option("display.width", 120)
print("=== per-tissue probe summary ===")
print(summary.to_string(index=False))
print(f"\nTOTAL: {len(sp)} blocks | {sp['is_hotspot'].sum()} hotspots (q_sup_trained<0.05)")
print(f"Global sup_trained_auc: median={sp['sup_trained_auc'].median():.3f}, "
      f"max={sp['sup_trained_auc'].max():.3f}")


=== per-tissue probe summary ===
tissue  n_blocks  n_hotspot  median_sup_auc  max_sup_auc  median_sex_auc  median_dose_rho  hotspot_pct
   bat         6          0          0.4815        0.558          0.6800           0.1225          0.0
 blood         8          4          0.7965        0.837          0.7960           0.3660         50.0
cortex        10          0          0.5850        0.698          0.5890           0.0615          0.0
 heart        12          0          0.6555        0.760          0.9740           0.2080          0.0
hippoc         7          0          0.5010        0.608          0.5290           0.0370          0.0
hypoth         8          1          0.7260        0.798          0.6765          -0.1380         12.5
kidney        11          0          0.5160        0.655          1.0000          -0.0380          0.0
 liver         8          0          0.4785        0.630          1.0000           0.1260          0.0
  lung         7          1          0.7

## FIG 1 — Supervised trained AUC distribution across the 151 analysable blocks

The histogram shows all 151 analysable block-level `sup_trained_auc` values.
Blocks with `q_sup_trained < 0.05` (exercise hotspots) are shaded darker.
The KDE curve overlaid on the AUC axis shows the shape of the full distribution.
The inset rug on the dose axis shows the distribution of `sup_dose_rho`.


In [3]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, matplotlib.gridspec as gridspec
from scipy.stats import gaussian_kde

from nb_config import GATE_DIR
GCI = str(GATE_DIR)
sp = pd.read_csv(f"{GCI}/subspace_probe.tsv", sep="\t")
q_values = pd.read_csv(
    f"{GCI}/corroboration_merged.tsv",
    sep="\t",
    usecols=["tissue", "cell_type", "q_sup_trained", "analysable"],
)
sp = sp.merge(q_values, on=["tissue", "cell_type"], how="left", validate="one_to_one")
sp = sp[sp["analysable"]].copy()   # §6 rule: blocks with median library < 1,000 are not analysed (no q value)
if sp["q_sup_trained"].isna().any():
    raise ValueError("Missing adjusted P values after merging Stage 11 result tables")
sp["is_hotspot"] = sp["q_sup_trained"] < 0.05

PALETTE = {
    "bat":"#D35400","blood":"#C0392B","cortex":"#8E44AD","heart":"#E74C3C",
    "hippoc":"#9B59B6","hypoth":"#6C3483","kidney":"#2980B9","liver":"#1ABC9C",
    "lung":"#27AE60","skmgn":"#F39C12","skmvl":"#E67E22","smlint":"#16A085",
    "ovary":"#EC407A","spleen":"#B7950B",
    "testes":"#5D6D7E","watsc":"#95A5A6",
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), constrained_layout=True)

# Left: AUC histogram
ax = axes[0]
hot = sp[sp["is_hotspot"]]["sup_trained_auc"].values
non = sp[~sp["is_hotspot"]]["sup_trained_auc"].values
bins = np.linspace(0.3, 1.0, 28)
ax.hist(non, bins=bins, color="#3498DB", alpha=0.65, label=f"Non-hotspot (n={len(non)})")
ax.hist(hot, bins=bins, color="#E74C3C", alpha=0.85, label=f"Hotspot q<0.05 (n={len(hot)})")

# KDE overlay
all_auc = sp["sup_trained_auc"].values
kde = gaussian_kde(all_auc, bw_method=0.35)
xs  = np.linspace(0.3, 1.0, 200)
ys  = kde(xs)
ax2 = ax.twinx()
ax2.plot(xs, ys, color="#2C3E50", lw=2.0, ls="--", alpha=0.7, label="KDE")
ax2.set_ylabel("Density", fontsize=9, color="#2C3E50")
ax2.tick_params(axis="y", labelcolor="#2C3E50")
ax2.set_ylim(0)

ax.axvline(0.65, color="#E74C3C", lw=1.4, ls=":", alpha=0.8, label="AUC = 0.65")
ax.axvline(0.5,  color="gray",    lw=1.0, ls="--", alpha=0.5)
ax.set_xlabel("Supervised trained AUC (PLS-1 OOF)", fontsize=10)
ax.set_ylabel("Block count", fontsize=10)
ax.set_title(f"Distribution of PLS-1 Trained AUC\n({len(sp)} analysable blocks, {sp['tissue'].nunique()} tissues)", fontsize=11)
lines1, labs1 = ax.get_legend_handles_labels()
lines2, labs2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labs1 + labs2, fontsize=8, loc="upper left")

# Right: dose rho distribution by tissue
ax = axes[1]
tissues = sorted(sp["tissue"].unique())
dose_data = [sp[sp["tissue"]==t]["sup_dose_rho"].values for t in tissues]
parts = ax.violinplot(dose_data, positions=range(len(tissues)), showmedians=True, widths=0.7)
for i, (body, t) in enumerate(zip(parts["bodies"], tissues)):
    body.set_facecolor(PALETTE.get(t, "#95A5A6"))
    body.set_alpha(0.7)
parts["cmedians"].set_color("#2C3E50")
parts["cbars"].set_color("#2C3E50")
ax.axhline(0, color="gray", lw=1.0, ls="--", alpha=0.6)
ax.set_xticks(range(len(tissues)))
ax.set_xticklabels([t.upper() for t in tissues], rotation=35, ha="right", fontsize=8)
ax.set_ylabel("Supervised dose Spearman ρ", fontsize=10)
ax.set_title("Ordinal Dose ρ per Tissue\n(positive = earlier weeks → lower expression)", fontsize=11)
ax.set_ylim(-1.05, 1.05)

plt.show()
print(f"hotspot count: {sp['is_hotspot'].sum()} / {len(sp)}")
print(f"sup_trained_auc: median={sp['sup_trained_auc'].median():.3f}, max={sp['sup_trained_auc'].max():.3f}")


hotspot count: 9 / 151
sup_trained_auc: median=0.605, max=0.887


## Supervised probe vs global (omnibus) detection

The global trace η² (`glob_trained`) is a permutation-tested MANOVA-like statistic that
tests whether any linear combination of dimensions separates trained vs control.
It is a less powerful estimator than PLS-1 because it treats all 768 dimensions equally
and does not learn the separation axis.

Comparing `glob_trained` to `sup_trained_auc` reveals the **power gain from supervision**:
blocks where the global measure is near zero but the supervised AUC is high indicate that
the exercise signal lives in a low-variance but informative subspace (a hallmark of transformer
CLS tokens where global variance is dominated by cell-type identity, not treatment).


In [4]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from scipy.stats import spearmanr

from nb_config import GATE_DIR
GCI = str(GATE_DIR)
sp = pd.read_csv(f"{GCI}/subspace_probe.tsv", sep="\t")
q_values = pd.read_csv(
    f"{GCI}/corroboration_merged.tsv",
    sep="\t",
    usecols=["tissue", "cell_type", "q_sup_trained", "analysable"],
)
sp = sp.merge(q_values, on=["tissue", "cell_type"], how="left", validate="one_to_one")
sp = sp[sp["analysable"]].copy()   # §6 rule: blocks with median library < 1,000 are not analysed (no q value)
if sp["q_sup_trained"].isna().any():
    raise ValueError("Missing adjusted P values after merging Stage 11 result tables")
sp["is_hotspot"] = sp["q_sup_trained"] < 0.05

PALETTE = {
    "bat":"#D35400","blood":"#C0392B","cortex":"#8E44AD","heart":"#E74C3C",
    "hippoc":"#9B59B6","hypoth":"#6C3483","kidney":"#2980B9","liver":"#1ABC9C",
    "lung":"#27AE60","skmgn":"#F39C12","skmvl":"#E67E22","smlint":"#16A085",
    "ovary":"#EC407A","spleen":"#B7950B",
    "testes":"#5D6D7E","watsc":"#95A5A6",
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), constrained_layout=True)

# Left: glob_trained vs sup_trained_auc
ax = axes[0]
for t in sorted(sp["tissue"].unique()):
    sub = sp[sp["tissue"] == t]
    m   = sub["is_hotspot"]
    ax.scatter(sub.loc[~m,"glob_trained"], sub.loc[~m,"sup_trained_auc"],
               c=PALETTE.get(t,"gray"), s=25, alpha=0.55, label=t.upper())
    ax.scatter(sub.loc[m, "glob_trained"], sub.loc[m, "sup_trained_auc"],
               c=PALETTE.get(t,"gray"), s=70, edgecolors="k", lw=1.2, zorder=5)

rho, pval = spearmanr(sp["glob_trained"], sp["sup_trained_auc"])
ax.axhline(0.65, color="#E74C3C", lw=1.2, ls=":", alpha=0.7, label="AUC hotspot threshold")
ax.set_xlabel("Global trace η² (trained)", fontsize=10)
ax.set_ylabel("Supervised AUC (PLS-1)", fontsize=10)
ax.set_title(f"Global vs Supervised Detection\nSpearman r={rho:.3f} (p={pval:.2e})", fontsize=11)
ax.legend(fontsize=7, ncol=2, loc="lower right")
ax.text(0.98, 0.03, "Hotspot (q<0.05)\nOther blocks",
        transform=ax.transAxes, ha="right", fontsize=8, color="#444")

# Right: per-dim max η² vs supervised AUC
ax = axes[1]
for t in sorted(sp["tissue"].unique()):
    sub = sp[sp["tissue"] == t]
    m   = sub["is_hotspot"]
    ax.scatter(sub.loc[~m,"pdmax_trained"], sub.loc[~m,"sup_trained_auc"],
               c=PALETTE.get(t,"gray"), s=25, alpha=0.55, label=t.upper())
    ax.scatter(sub.loc[m,"pdmax_trained"],  sub.loc[m,"sup_trained_auc"],
               c=PALETTE.get(t,"gray"), s=70, edgecolors="k", lw=1.2, zorder=5)

rho2, pval2 = spearmanr(sp["pdmax_trained"], sp["sup_trained_auc"])
ax.axhline(0.65, color="#E74C3C", lw=1.2, ls=":", alpha=0.7)
ax.set_xlabel("Per-dim max η² (trained)", fontsize=10)
ax.set_ylabel("Supervised AUC (PLS-1)", fontsize=10)
ax.set_title(f"Per-dim Max vs Supervised Detection\nSpearman r={rho2:.3f} (p={pval2:.2e})", fontsize=11)
ax.legend(fontsize=7, ncol=2, loc="lower right")

plt.show()
print(f"glob vs sup: r={rho:.3f}  pdmax vs sup: r={rho2:.3f}")


glob vs sup: r=0.851  pdmax vs sup: r=0.569


## Corroboration: PLS-1 supervised probe vs Augur random-forest

Stage 9 produced canonical Augur AUC values (random-forest classifier, `augur_embed_trained`) per block
using the same GeneCompass embeddings. The Augur RF and PLS-1 probe are **independent estimators**
of the same quantity (trained-vs-control discriminability), so their agreement is a
pre-registration corroboration check.

`corroboration_merged.tsv` joins the probe results with the PCA control's PLS-1 AUCs by representation
(`auc_embed`, `auc_pca`, `auc_genes`) and the Augur columns (`augur_embed_sex`, `augur_embed_trained`, ...).
Note that `auc_embed` is the PLS-1 probe again (under the gate method it equals `sup_trained_auc`), so the
corroboration uses `augur_embed_trained`. TESTES and OVARY (single-sex; 5 control pseudo-cells per arm) have no Augur score, leaving 130 analysable blocks.

A high Spearman r between `augur_embed_trained` (Augur-RF) and `sup_trained_auc` (PLS-1) means:
- Both methods find the same blocks to be discriminative.
- The embedding signal is not an artifact of one estimator's inductive bias.

**Expected:** r ≥ 0.80 (pre-registered threshold). On all 16 tissues r = 0.785, under it (refs4 0.788; July references 0.822).


In [5]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from scipy.stats import spearmanr

from nb_config import GATE_DIR
GCI = str(GATE_DIR)
cm  = pd.read_csv(f"{GCI}/corroboration_merged.tsv", sep="\t")
cm = cm[cm["analysable"]].copy()
cm["is_hotspot"] = cm["q_sup_trained"] < 0.05
# canonical Augur (random forest) on the embedding; TESTES and OVARY (single-sex) have too few control cells -> NaN, dropped
cm = cm.dropna(subset=["augur_embed_trained"]).copy()
cm["augur_rf"] = cm["augur_embed_trained"]

PALETTE = {
    "bat":"#D35400","blood":"#C0392B","cortex":"#8E44AD","heart":"#E74C3C",
    "hippoc":"#9B59B6","hypoth":"#6C3483","kidney":"#2980B9","liver":"#1ABC9C",
    "lung":"#27AE60","skmgn":"#F39C12","skmvl":"#E67E22","smlint":"#16A085",
    "ovary":"#EC407A","spleen":"#B7950B",
    "testes":"#5D6D7E","watsc":"#95A5A6",
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), constrained_layout=True)

# Left: Augur RF vs PLS-1 sup_trained_auc
ax = axes[0]
for t in sorted(cm["tissue"].unique()):
    sub = cm[cm["tissue"] == t]
    m   = sub["is_hotspot"]
    ax.scatter(sub.loc[~m,"augur_rf"], sub.loc[~m,"sup_trained_auc"],
               c=PALETTE.get(t,"gray"), s=25, alpha=0.55, label=t.upper())
    ax.scatter(sub.loc[m,"augur_rf"],  sub.loc[m,"sup_trained_auc"],
               c=PALETTE.get(t,"gray"), s=70, edgecolors="k", lw=1.2, zorder=5)

rho, pval = spearmanr(cm["augur_rf"], cm["sup_trained_auc"])

# identity reference line
lo = min(cm["augur_rf"].min(), cm["sup_trained_auc"].min()) - 0.02
hi = max(cm["augur_rf"].max(), cm["sup_trained_auc"].max()) + 0.02
ax.plot([lo, hi],[lo, hi], "k--", alpha=0.3, lw=1.0)
ax.set_xlabel("Augur RF AUC (embedding)", fontsize=10)
ax.set_ylabel("PLS-1 supervised AUC (Stage 11)", fontsize=10)
ax.set_title(f"Corroboration: Augur-RF vs PLS-1\nSpearman r={rho:.3f} (p={pval:.2e}, n={len(cm)})", fontsize=11)
ax.legend(fontsize=7, ncol=2, loc="upper left")

# Right: Augur vs PLS-1 for hotspots only (zoomed)
ax = axes[1]
hot = cm[cm["is_hotspot"]]
for t in sorted(hot["tissue"].unique()):
    sub = hot[hot["tissue"] == t]
    ax.scatter(sub["augur_rf"], sub["sup_trained_auc"],
               c=PALETTE.get(t,"gray"), s=90, edgecolors="k", lw=1.0,
               label=t.upper(), zorder=5)

if len(hot) > 1:
    rho2, pval2 = spearmanr(hot["augur_rf"], hot["sup_trained_auc"])
    lo2 = min(hot["augur_rf"].min(), hot["sup_trained_auc"].min()) - 0.02
    hi2 = max(hot["augur_rf"].max(), hot["sup_trained_auc"].max()) + 0.02
    ax.plot([lo2,hi2],[lo2,hi2], "k--", alpha=0.3, lw=1.0)
    ax.set_title(f"Hotspot blocks only (n={len(hot)})\nSpearman r={rho2:.3f}", fontsize=11)
else:
    ax.set_title(f"Hotspot blocks (n={len(hot)})", fontsize=11)

ax.set_xlabel("Augur RF AUC (embed)", fontsize=10)
ax.set_ylabel("PLS-1 supervised AUC", fontsize=10)
ax.legend(fontsize=8, loc="upper left")

plt.show()
print(f"overall corroboration r={rho:.3f}, hotspot n={len(hot)}")


overall corroboration r=0.785, hotspot n=9


## Per-tissue hotspot profile

Summarizing which tissues contribute hotspots under three progressively conservative criteria:
1. **Augur-RF**: `augur_embed_trained > 0.65` (Stage 9 threshold)
2. **PLS-1 nominal**: `p_sup_trained < 0.05`
3. **PLS-1 BH-adjusted**: `q_sup_trained < 0.05` (pre-registered primary criterion)

The gap between (1) and (3) reflects multiple-testing burden across the 151 analysable blocks.
Tissues with many blocks (blood, cortex) face a higher FDR penalty than single-block tissues.


In [6]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt

from nb_config import GATE_DIR
GCI = str(GATE_DIR)
sp = pd.read_csv(f"{GCI}/subspace_probe.tsv", sep="\t")
q_values = pd.read_csv(
    f"{GCI}/corroboration_merged.tsv",
    sep="\t",
    usecols=["tissue", "cell_type", "q_sup_trained", "analysable"],
)
sp = sp.merge(q_values, on=["tissue", "cell_type"], how="left", validate="one_to_one")
sp = sp[sp["analysable"]].copy()   # §6 rule: blocks with median library < 1,000 are not analysed (no q value)
if sp["q_sup_trained"].isna().any():
    raise ValueError("Missing adjusted P values after merging Stage 11 result tables")
cm  = pd.read_csv(f"{GCI}/corroboration_merged.tsv", sep="\t")
cm = cm[cm["analysable"]].copy()

PALETTE = {
    "bat":"#D35400","blood":"#C0392B","cortex":"#8E44AD","heart":"#E74C3C",
    "hippoc":"#9B59B6","hypoth":"#6C3483","kidney":"#2980B9","liver":"#1ABC9C",
    "lung":"#27AE60","skmgn":"#F39C12","skmvl":"#E67E22","smlint":"#16A085",
    "ovary":"#EC407A","spleen":"#B7950B",
    "testes":"#5D6D7E","watsc":"#95A5A6",
}
tissues = sorted(sp["tissue"].unique())
n_blocks   = [len(sp[sp["tissue"]==t]) for t in tissues]
n_hot_q    = [(sp[sp["tissue"]==t]["q_sup_trained"] < 0.05).sum() for t in tissues]
n_hot_pnom = [(sp[sp["tissue"]==t]["p_sup_trained"] < 0.05).sum() for t in tissues]
n_hot_aug  = [(cm[cm["tissue"]==t]["augur_embed_trained"] > 0.65).sum() for t in tissues]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), constrained_layout=True)

# Left: stacked bar: n_blocks decomposed into hotspot tiers
ax = axes[0]
x = np.arange(len(tissues))
w = 0.55
base = np.zeros(len(tissues))
q_arr    = np.array(n_hot_q)
pnom_arr = np.array(n_hot_pnom)
aug_arr  = np.array(n_hot_aug)
other    = np.array(n_blocks) - q_arr

ax.bar(x, other,    w, bottom=base,   color="#D5D8DC", label="Non-hotspot")
ax.bar(x, q_arr,    w, bottom=other,  color="#E74C3C", label="BH q<0.05 (primary)")

# overlay dots for nominal and Augur
for i in range(len(tissues)):
    ax.scatter(x[i], other[i]+q_arr[i]+0.3, marker="^", color="#E74C3C", s=0, zorder=6)

ax.set_xticks(x)
ax.set_xticklabels([t.upper() for t in tissues], rotation=40, ha="right", fontsize=8)
ax.set_ylabel("Block count", fontsize=10)
ax.set_title("Hotspot blocks per tissue\n(primary: q_sup_trained < 0.05)", fontsize=11)
ax.legend(fontsize=9, loc="upper right")

# annotate hotspot count
for i in range(len(tissues)):
    if q_arr[i] > 0:
        ax.text(x[i], other[i]+q_arr[i]+0.1, str(int(q_arr[i])),
                ha="center", va="bottom", fontsize=9, fontweight="bold", color="#C0392B")

# Right: median sup_trained_auc ± IQR per tissue, ordered by median
ax = axes[1]
medians  = [sp[sp["tissue"]==t]["sup_trained_auc"].median() for t in tissues]
q25s     = [sp[sp["tissue"]==t]["sup_trained_auc"].quantile(0.25) for t in tissues]
q75s     = [sp[sp["tissue"]==t]["sup_trained_auc"].quantile(0.75) for t in tissues]
order    = np.argsort(medians)[::-1]
t_ord    = [tissues[i] for i in order]
med_ord  = [medians[i] for i in order]
lo_ord   = [medians[i] - q25s[i] for i in order]
hi_ord   = [q75s[i] - medians[i] for i in order]
colors   = [PALETTE.get(t,"gray") for t in t_ord]

for j, (t, med, lo, hi, col) in enumerate(zip(t_ord, med_ord, lo_ord, hi_ord, colors)):
    ax.barh(j, med, xerr=[[lo],[hi]], color=col, ecolor="#2C3E50", capsize=4,
            height=0.6, alpha=0.8)
    ax.text(0.31, j, t.upper(), va="center", ha="left", fontsize=8.5, color="#2C3E50")

ax.axvline(0.65, color="#E74C3C", lw=1.3, ls=":", alpha=0.8, label="AUC=0.65")
ax.axvline(0.5,  color="gray",    lw=1.0, ls="--", alpha=0.5, label="AUC=0.5 (chance)")
ax.set_yticks(range(len(tissues)))
ax.set_yticklabels([])
ax.set_xlabel("Supervised trained AUC (median ± IQR)", fontsize=10)
ax.set_title("PLS-1 trained AUC profile per tissue\n(ordered by median)", fontsize=11)
ax.legend(fontsize=9)

plt.show()
for t, med, nq, nb in zip(tissues, medians, n_hot_q, n_blocks):
    print(f"  {t:<8}: {nb:2d} blocks, {int(nq):2d} hotspot, med_auc={med:.3f}")


  bat     :  6 blocks,  0 hotspot, med_auc=0.481
  blood   :  8 blocks,  4 hotspot, med_auc=0.796
  cortex  : 10 blocks,  0 hotspot, med_auc=0.585
  heart   : 12 blocks,  0 hotspot, med_auc=0.655
  hippoc  :  7 blocks,  0 hotspot, med_auc=0.501
  hypoth  :  8 blocks,  1 hotspot, med_auc=0.726
  kidney  : 11 blocks,  0 hotspot, med_auc=0.516
  liver   :  8 blocks,  0 hotspot, med_auc=0.478
  lung    :  7 blocks,  1 hotspot, med_auc=0.781
  ovary   : 15 blocks,  0 hotspot, med_auc=0.731
  skmgn   :  8 blocks,  1 hotspot, med_auc=0.704
  skmvl   :  8 blocks,  2 hotspot, med_auc=0.762
  smlint  : 13 blocks,  0 hotspot, med_auc=0.605
  spleen  : 12 blocks,  0 hotspot, med_auc=0.547
  testes  :  6 blocks,  0 hotspot, med_auc=0.480
  watsc   : 12 blocks,  0 hotspot, med_auc=0.499


## Stage Summary

Stage 11 evaluates **151** analysable tissue-by-cell-type blocks (of 197; the §6 rule skips 46 near-empty ones) with held-out PLS-1 prediction and permutation testing on the v2 embeddings of the all-tissue pseudo-cells. **Nine** blocks pass the BH-adjusted supervised-probe criterion: blood 4 (classical monocytes, erythroid cells, megakaryocytes and platelets, neutrophils), skmvl 2 (satellite cells, vascular smooth muscle cells), skmgn 1 (tenocytes), lung 1 (vascular smooth muscle cells) and hypoth 1 (oligodendrocyte precursor cells).

Compared with refs4, 14 tissues (`deconvolution/ALL_TISSUE_RERUN_RESULTS.md`):

- **Hotspots.** 9 against 11. LUNG endothelial and mesothelial cells drop out (q 0.0440 → 0.0535) with unchanged AUCs and p values: the gate is one BH family, and the 32 new SPLEEN and OVARY blocks (none significant) enlarge it. Every q rises by about 1.2; there is no new hotspot. Statements about LUNG endothelial or mesothelial cells as hotspots hold for refs4 (14 tissues) only.
- **SPLEEN and OVARY.** No block reaches q < 0.05. OVARY's best is smooth muscle cells (AUC 0.845, q 0.081); with 24 samples of one sex its p values have a coarse floor. SPLEEN's best is red-pulp macrophages (AUC 0.663, q 0.26). Both are read differentially only.
- **Dose and sex.** Median dose ρ 0.073 and median sex AUC 0.804 over the analysable blocks (OVARY and TESTES have no sex AUC).
- **Augur.** Canonical Augur gives an AUC of at least 0.73 in all 9 hotspots; its block AUCs follow the probe's at Spearman 0.785 over 130 blocks (OVARY and TESTES have no Augur score), under the registered 0.80.
- **Embedding vs PCA.** The embedding's AUC beats PCA-50 of the same pseudo-cells in 82 of 151 blocks (median +0.009, Wilcoxon p 0.15), no longer significant (refs4: 71 of 124, p 0.013).


## Reproducibility Notes

The all-tissue gate ran in job **12108627** (2026-11-01; the 14 refs4 tissues are linked in, and the refs4 gate was job 12042041). Each block used a one-component PLS probe, stratified five-fold held-out predictions over 10 independent splits, and 10,000 label permutations. BH-FDR correction was applied jointly across the 151 analysable blocks' `p_sup_trained` values; sensitivity tables at library thresholds 100 and 0 are `corroboration_merged_minlib100.tsv` and `corroboration_merged_minlib0.tsv`.

```bash
python deconvolution/subspace_probe.py \
    --gc-root data/deconvolution/genecompass_input_v2_refs4all \
    --out data/deconvolution/genecompass_input_v2_refs4all/gate_rcv10/subspace_probe.tsv \
    --perms 10000 --folds 5 --repeats 10 --jobs 48
python deconvolution/augur_prep.py --gc-root data/deconvolution/genecompass_input_v2_refs4all --pca 50 \
    --perms 10000 --folds 5 --repeats 10 --jobs 48 --out <scratch> \
    --ctl-out data/deconvolution/genecompass_input_v2_refs4all/gate_rcv10/pca_control.tsv
python deconvolution/corroborate_summary.py --gc-root data/deconvolution/genecompass_input_v2_refs4all/gate_rcv10
```

Folds and permutation nulls are seeded per block (stable across runs). `corroboration_merged.tsv` joins the supervised probe with the PCA control and canonical Augur by tissue and cell type.

In [7]:
import pandas as pd, hashlib, pathlib

from nb_config import GATE_DIR as GCI
files = ["subspace_probe.tsv","corroboration_merged.tsv","pheno_merge_test.tsv"]

for fn in files:
    p = GCI / fn if fn != "pheno_merge_test.tsv" else GCI.parent / fn
    md5 = hashlib.md5(p.read_bytes()).hexdigest()
    nrows = len(open(p).readlines()) - 1
    print(f"  {fn}: {nrows} data rows | MD5={md5}")

sp = pd.read_csv(GCI / "subspace_probe.tsv", sep="\t")
q_values = pd.read_csv(GCI / "corroboration_merged.tsv", sep="\t", usecols=["tissue", "cell_type", "q_sup_trained", "analysable"])
sp = sp.merge(q_values, on=["tissue", "cell_type"], how="left", validate="one_to_one")
sp = sp[sp["analysable"]].copy()   # §6 rule: blocks with median library < 1,000 are not analysed (no q value)
print(f"\n  analysable blocks: {len(sp)}")
print(f"  hotspot (q<0.05): {(sp['q_sup_trained']<0.05).sum()}")
print(f"  nominal sig (p<0.05): {(sp['p_sup_trained']<0.05).sum()}")
print(f"  sup_trained_auc > 0.65: {(sp['sup_trained_auc']>0.65).sum()}")
print(f"  sup_sex_auc median: {sp['sup_sex_auc'].median():.3f}  (positive control)")


  subspace_probe.tsv: 197 data rows | MD5=6e1c176e0dc7c7f1c593d96e064dbe1a
  corroboration_merged.tsv: 197 data rows | MD5=4ce1becb8f3902e4ddbf66b685197007
  pheno_merge_test.tsv: 197 data rows | MD5=a96af33ff4242649155f915dff8e6daf

  analysable blocks: 151
  hotspot (q<0.05): 9
  nominal sig (p<0.05): 36
  sup_trained_auc > 0.65: 58
  sup_sex_auc median: 0.804  (positive control)
